## Setup

In [8]:
## INSTRUCTION: DEPENDING ON WHO IS USING THIS, PLEASE COMMENT OUT THE OTHER SYS.PATH.APPEND THAT DOES NOT RELATE TO YOUR DEVICE. 

import sys
sys.path.append('/Users/annaglass/capstone/capstone')
#sys.path.append('/Users/jasmi/capstone')
#sys.path.append('/Users/moham/Downloads/New folder/capstone')
#sys.path.append('/Users/marks/OneDrive/Documents/Georgetown/capstone)
#sys.path.append('/Users\polivetti\capstone\capstone')

In [9]:
import pandas as pd
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import random

### Comparison in Datasets

In [9]:
df = pd.read_csv('../data/final_dataset_sampled.csv')

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 379225 entries, 0 to 379224
Data columns (total 23 columns):
 #   Column                Non-Null Count   Dtype 
---  ------                --------------   ----- 
 0   tag_name              379225 non-null  object
 1   article_id            379225 non-null  int64 
 2   source_feed_name      379225 non-null  object
 3   load_date             379225 non-null  object
 4   feed_name             379225 non-null  object
 5   author_name           379225 non-null  object
 6   source_unique_id      379225 non-null  object
 7   source_type_name      379225 non-null  object
 8   channel_name          379225 non-null  object
 9   genre                 379225 non-null  object
 10  publisher_name        379225 non-null  object
 11  publication_name      379225 non-null  object
 12  circulation_size      379225 non-null  int64 
 13  sentiment_score       379225 non-null  int64 
 14  source_type           379225 non-null  object
 15  sentiment_band   

In [3]:
df1 = pd.read_csv('../data/processed/text_processed_data.csv')

In [16]:
df1.head()

,tag_name,article_id,source_feed_name,load_date,feed_name,headline,article_body,author_name,source_unique_id,source_type_name,...,vipr_weight_log,hit_strength_log,circulation_size_log,vipr_score_clip,processed_headline,processed_body,processed_text,headline_token_count,body_token_count,token_count
0,Public Health,18083066104,regionals-web,2024-03-25,opoint,"Dairies in Texas, Kansas, New Mexico Report Ca...",Some vendors may process your personal data o...,Chris Clayton,123878-67182,Regional News,...,5.993961,0.693147,6.834109,-9600.0,dairy texas kansa new mexico report cattle inf...,vendor may process personal data basis legitim...,dairy texas kansa new mexico report cattle inf...,10,571,581
1,Public Health,18083066685,regionals-web,2024-03-25,opoint,Lawmakers call for an independent report on NY...,"ALBANY, N.Y. (NEWS10)—At the New York State C...",Jamie DeLine,31229-477204,Regional News,...,6.530878,1.098612,12.459982,-4795.0,lawmaker call independent report ny pandemic r...,albany news new york state capitol monday lost...,lawmaker call independent report ny pandemic r...,7,237,244
2,Public Health,18083068112,regionals-web,2024-03-25,opoint,‘Yellowstone' star Forrie J. Smith says he was...,Forrie J. Smith has been vocal about his publ...,Katherine Itoh,21944-341746,Regional News,...,7.170888,1.386294,14.320644,-3900.0,yellowstone star forrie smith say kicked fligh...,forrie smith vocal public health belief skippe...,yellowstone star forrie smith say kicked fligh...,12,227,239
3,Public Health,18083069120,trade-web,2024-03-25,opoint,Free RSV immunisation program for Queensland i...,JOINT STATEMENT Premier The Honourable St...,uncredited,156350-188089,Trade News,...,7.791110,2.639057,10.310917,16926.0,free rsv immunisation program queensland infan...,joint statement premier honourable steven mile...,free rsv immunisation program queensland infan...,8,571,579
4,Public Health,18083070124,consumer,2024-03-25,opoint,Subtle Nods You Didn't Notice In Kate Middleto...,Kate Middleton took complete control of ann...,Lauren Waters,263959-59381,Consumer,...,4.804021,0.693147,15.141498,1089.0,subtle nod notice kate middleton cancer announ...,kate middleton took complete control announcin...,subtle nod notice kate middleton cancer announ...,7,360,367


In [17]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 946261 entries, 0 to 946260
Data columns (total 32 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   tag_name              946261 non-null  object 
 1   article_id            946261 non-null  int64  
 2   source_feed_name      946261 non-null  object 
 3   load_date             946261 non-null  object 
 4   feed_name             946261 non-null  object 
 5   headline              946261 non-null  object 
 6   article_body          946261 non-null  object 
 7   author_name           946261 non-null  object 
 8   source_unique_id      946261 non-null  object 
 9   source_type_name      946261 non-null  object 
 10  channel_name          946261 non-null  object 
 11  genre                 946261 non-null  object 
 12  publisher_name        946261 non-null  object 
 13  publication_name      946261 non-null  object 
 14  circulation_size      946261 non-null  int64  
 15  

### Sampling

In [4]:
# network_sampler.py
from typing import Optional, Sequence
import pandas as pd
import numpy as np

def sample_for_network(
    df: pd.DataFrame,
    source_type_col: str = "source_type_name",
    influence_cols: Sequence[str] = ("vipr_score",),   # e.g., ("vipr_score","engagement")
    protect_quantile: float = 0.90,   # keep top 10% per type by influence
    min_per_type: int = 2000,         # guarantee coverage of every type
    max_per_type: Optional[int] = None,       # cap per type (e.g., 15000) or None
    target_n: Optional[int] = 100_000,        # total target size or None for “as is”
    weight_col: Optional[str] = "vipr_score", # used only when proportional downsampling
    random_state: int = 42
) -> pd.DataFrame:
    """
    Network-aware sampler:
    1) Protects high-influence rows per source type (top quantile of influence_cols).
    2) Ensures a minimum number per type (fills by highest influence if needed).
    3) Optionally caps max per type.
    4) Optionally downsamples to a global target_n proportional to weights.

    Returns a sampled DataFrame that's smaller but preserves structural signals.
    """
    rng = np.random.default_rng(random_state)
    df = df.copy()

    # ---- folded influence score (supports multiple cols; rank-average is scale-robust) ----
    if len(influence_cols) == 1:
        folded_influence = pd.to_numeric(df[influence_cols[0]], errors="coerce").fillna(0.0)
    else:
        ranks = [
            pd.to_numeric(df[c], errors="coerce").fillna(0.0).rank(pct=True)
            for c in influence_cols
        ]
        folded_influence = pd.concat(ranks, axis=1).mean(axis=1)
    df["_folded_influence"] = folded_influence

    # ---- 1) Protect: keep top quantile per type by folded influence ----
    def _protect_group(g: pd.DataFrame) -> pd.Series:
        q = g["_folded_influence"].quantile(protect_quantile)
        return g["_folded_influence"] >= q

    protect_mask = (
        df.groupby(source_type_col, group_keys=False)
          .apply(_protect_group)
          .reindex(df.index)
    )
    protected = df[protect_mask]
    unprotected = df[~protect_mask]

    # ---- 2) Ensure minimum per type: top-up with highest influence ----
    need_rows = []
    protected_counts = protected[source_type_col].value_counts()

    for stype, group in df.groupby(source_type_col):
        have = int(protected_counts.get(stype, 0))
        if have < min_per_type:
            deficit = min_per_type - have
            pool = unprotected[unprotected[source_type_col] == stype].sort_values(
                "_folded_influence", ascending=False
            )
            if deficit > 0 and not pool.empty:
                need_rows.append(pool.head(deficit))

    topped_up = pd.concat(need_rows, axis=0) if need_rows else df.iloc[0:0]
    current = pd.concat([protected, topped_up], axis=0).drop_duplicates()

    if not topped_up.empty:
        unprotected = unprotected.drop(index=topped_up.index, errors="ignore")

    # ---- 3) Cap max per type (if set) keeping protected first ----
    if max_per_type is not None:
        keep_chunks = []
        for stype, group in current.groupby(source_type_col):
            if len(group) <= max_per_type:
                keep_chunks.append(group)
                continue

            g_prot = group[group.index.isin(protected.index)]
            g_rest = group[~group.index.isin(g_prot.index)]
            remaining = max(max_per_type - len(g_prot), 0)

            if remaining > 0 and len(g_rest) > 0:
                g_rest_sampled = g_rest.sample(n=min(remaining, len(g_rest)), random_state=random_state)
                keep_chunks.extend([g_prot, g_rest_sampled])
            else:
                # if protected alone exceed cap, trim protected deterministically
                keep_chunks.append(g_prot.sample(n=max_per_type, random_state=random_state))

        current = pd.concat(keep_chunks, axis=0)

    # ---- 4) Proportional extras (if target_n larger than current) ----
    if max_per_type is None and target_n is not None and len(current) < target_n:
        remaining_n = target_n - len(current)
        pool = df.drop(index=current.index, errors="ignore")

        # weights if available
        if weight_col is not None and weight_col in pool.columns:
            w = pd.to_numeric(pool[weight_col], errors="coerce").fillna(0.0)
        else:
            w = None

        by_type_target = df[source_type_col].value_counts(normalize=True)
        extras = []
        pool_by_type = pool.groupby(source_type_col)

        for stype, frac in by_type_target.items():
            want = int(round(frac * remaining_n))
            if want <= 0 or stype not in pool_by_type.groups:
                continue
            g = pool_by_type.get_group(stype)
            if len(g) == 0:
                continue

            if w is not None:
                gw = w.loc[g.index]
                s = float(gw.sum())
                if s > 0:
                    p = (gw / s).to_numpy()
                    take = min(want, len(g))
                    idx = rng.choice(g.index.to_numpy(), size=take, replace=False, p=p)
                    extras.append(g.loc[idx])
                else:
                    extras.append(g.sample(n=min(want, len(g)), random_state=random_state))
            else:
                extras.append(g.sample(n=min(want, len(g)), random_state=random_state))

        if extras:
            current = pd.concat([current] + extras, axis=0).drop_duplicates()

    # ---- 5) Global trim to target_n (keep all protected first) ----
    if target_n is not None and len(current) > target_n:
        prot_mask = current.index.isin(protected.index)
        keep_prot = current[prot_mask]
        keep_rest = current[~prot_mask]

        remaining = max(target_n - len(keep_prot), 0)
        if remaining < len(keep_rest):
            if weight_col is not None and weight_col in keep_rest.columns:
                wr = pd.to_numeric(keep_rest[weight_col], errors="coerce").fillna(0.0)
                s = float(wr.sum())
                if s > 0:
                    p = (wr / s).to_numpy()
                    sampled_rest = keep_rest.sample(n=remaining, random_state=random_state, weights=p)
                else:
                    sampled_rest = keep_rest.sample(n=remaining, random_state=random_state)
            else:
                sampled_rest = keep_rest.sample(n=remaining, random_state=random_state)
            current = pd.concat([keep_prot, sampled_rest], axis=0)
        else:
            current = pd.concat([keep_prot, keep_rest], axis=0).head(target_n)

    return current.drop(columns=["_folded_influence"])


def report_sampling_stats(
    full_df: pd.DataFrame,
    sample_df: pd.DataFrame,
    source_type_col: str = "source_type_name",
    influence_col: str = "vipr_score",
    top_k: int = 10
) -> None:
    """Quick sanity checks useful for network analysis."""
    def dist(x, col):
        c = x[col].value_counts(normalize=True).rename("share")
        n = x[col].value_counts().rename("count")
        return pd.concat([n, c.round(4)], axis=1)

    print("=== Size ===", {"full": len(full_df), "sample": len(sample_df)})
    print("\n=== Distribution by source type (full) ===")
    print(dist(full_df, source_type_col))
    print("\n=== Distribution by source type (sample) ===")
    print(dist(sample_df, source_type_col))
    print("\n=== Top sources by influence (full vs sample) ===")
    for name, df_ in [("full", full_df), ("sample", sample_df)]:
        cols = [c for c in ["publisher_name", "publication_name", source_type_col] if c in df_.columns]
        top = df_.nlargest(top_k, influence_col)[[influence_col] + cols]
        print(f"\nTop {top_k} ({name}):")
        print(top.reset_index(drop=True))

In [5]:
df_sample = sample_for_network(
    df1,
    source_type_col="source_type_name",
    influence_cols=("vipr_score",),   # or ("vipr_score","engagement")
    protect_quantile=0.90,
    min_per_type=2000,
    max_per_type=15000,               # or None
    target_n=300_000,
    weight_col="vipr_score",
    random_state=42
)

report_sampling_stats(df1, df_sample)

/var/folders/mt/qyhs8mhj6bd77bs78hd65s7w0000gn/T/ipykernel_16506/893312835.py:46: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.groupby(source_type_col, group_keys=False)


=== Size === {'full': 946261, 'sample': 76728}

=== Distribution by source type (full) ===
                   count   share
source_type_name                
Regional News     411449  0.4348
Radio             121455  0.1284
General News      100963  0.1067
Consumer           93556  0.0989
Trade News         62028  0.0656
TV                 56268  0.0595
Blog               26955  0.0285
Wires              17487  0.0185
National News      16747  0.0177
Owned Media        14220  0.0150
Government         10332  0.0109
Social Comments     7720  0.0082
Twitter             3534  0.0037
Video               2011  0.0021
Podcast             1463  0.0015
Forum                 70  0.0001
Unknown                3  0.0000

=== Distribution by source type (sample) ===
                  count   share
source_type_name               
Regional News     15000  0.1955
Radio             12149  0.1583
General News      10100  0.1316
Consumer           9376  0.1222
Trade News         6243  0.0814
TV          

In [6]:
df_sample.info()

<class 'pandas.core.frame.DataFrame'>
Index: 76728 entries, 569 to 388003
Data columns (total 32 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   tag_name              76728 non-null  object 
 1   article_id            76728 non-null  int64  
 2   source_feed_name      76728 non-null  object 
 3   load_date             76728 non-null  object 
 4   feed_name             76728 non-null  object 
 5   headline              76728 non-null  object 
 6   article_body          76728 non-null  object 
 7   author_name           76728 non-null  object 
 8   source_unique_id      76728 non-null  object 
 9   source_type_name      76728 non-null  object 
 10  channel_name          76728 non-null  object 
 11  genre                 76728 non-null  object 
 12  publisher_name        76728 non-null  object 
 13  publication_name      76728 non-null  object 
 14  circulation_size      76728 non-null  int64  
 15  sentiment_score      

In [7]:
df_sample.head()

,tag_name,article_id,source_feed_name,load_date,feed_name,headline,article_body,author_name,source_unique_id,source_type_name,...,vipr_weight_log,hit_strength_log,circulation_size_log,vipr_score_clip,processed_headline,processed_body,processed_text,headline_token_count,body_token_count,token_count
569,Public Health,18084602326,social-blog,2024-03-26,moreover-social,"Job postings: temporary positions, North Carol...","From the inbox, three temporary positions wit...",Chemjobber,1642267952860,Blog,...,6.621406,1.945910,6.831954,19500.0,job posting temporary position north carolina ...,inbox three temporary position north carolina ...,job posting temporary position north carolina ...,11,129,140
1014,Public Health,18085578718,social-blog,2024-03-26,moreover-social,Henry County Public Health Sensory Friendly Im...,Share Tweet Email Henry County Public Health ...,Yorke Prough,1642403692198,Blog,...,6.685861,2.079442,7.035269,16800.0,henry county public health sensory friendly im...,share tweet email henry county public health m...,henry county public health sensory friendly im...,9,151,160
1092,Public Health,18085707034,moreover_social,2024-03-26,web-rss-scraped,Sodium intake and cause-specific mortality amo...,<p><strong>About The Study:</strong>&nbsp;In t...,uncredited,https://www.eurekalert.org/news-releases/1038803,Blog,...,7.170888,1.609438,6.716595,14300.0,sodium intake cause specific mortality among p...,strong study strong nbsp cohort study low inco...,sodium intake cause specific mortality among p...,12,220,232
1368,Public Health,18086223321,social-blog,2024-03-26,moreover-social,Fisher-Titus nationally recognized for quality...,Anytime I see 'care' in any stroke press rel...,oc1dean,1642484470735,Blog,...,4.897840,0.693147,7.062192,10773.0,fisher titus nationally recognized quality car...,anytime see care stroke press release know str...,fisher titus nationally recognized quality car...,9,402,411
1383,Public Health,18086253120,moreover_social,2024-03-26,opoint,"Working together for a healthier, safer world:...","Geneva [Switzerland], March 26 (ANI/WAM): T...",uncredited,181740-71914,Blog,...,7.176255,1.791759,5.808142,52280.0,working together healthier safer world ipu ren...,geneva switzerland march ani wam inter parliam...,working together healthier safer world ipu ren...,11,291,302


### Top Words Sampled Dataset

In [10]:
sdf = pd.read_csv('../data/final_dataset_sampled.csv')

In [11]:
sdf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 379225 entries, 0 to 379224
Data columns (total 23 columns):
 #   Column                Non-Null Count   Dtype 
---  ------                --------------   ----- 
 0   tag_name              379225 non-null  object
 1   article_id            379225 non-null  int64 
 2   source_feed_name      379225 non-null  object
 3   load_date             379225 non-null  object
 4   feed_name             379225 non-null  object
 5   author_name           379225 non-null  object
 6   source_unique_id      379225 non-null  object
 7   source_type_name      379225 non-null  object
 8   channel_name          379225 non-null  object
 9   genre                 379225 non-null  object
 10  publisher_name        379225 non-null  object
 11  publication_name      379225 non-null  object
 12  circulation_size      379225 non-null  int64 
 13  sentiment_score       379225 non-null  int64 
 14  source_type           379225 non-null  object
 15  sentiment_band   